# EfficientNet-Lite4 → INT8 TFLite for Arduino UNO Q

Fine-tunes `tf_efficientnet_lite4` on the `data/{train,val,test}` ImageFolder prepared by `data_preparation.ipynb` and exports an 8-bit-integer `.tflite` file suitable for edge deployment on the Arduino UNO Q (quad-core Linux SoC, no GPU).

**Pipeline**

1. Load the three splits via `tf.keras.utils.image_dataset_from_directory` (class names come from the live directory listing — the legacy `outputs/class_names.json` is stale).
2. Build a Keras model: EfficientNet-Lite4 feature extractor from TF Hub + a Dense classifier head.
3. Float fine-tune — a short head warmup, then the full backbone.
4. Quantization-aware training (best-effort) — try `tfmot.quantization.keras.quantize_model`; fall back to head-only fake-quant if the hub backbone is opaque to tfmot (expected).
5. Full-integer TFLite conversion with a representative training subset → `outputs/efficientnet_lite4_int8.tflite`.
6. Run the exported TFLite on the held-out test split and compare accuracy vs. the Keras float model.

**Why EfficientNet-Lite**: ReLU6 + no squeeze-and-excitation blocks make it quantization-friendly. Weights are ImageNet-1k pretrained (ported from the official Tensorflow/TPU release). Lite4 is the largest Lite variant — best accuracy, still ~3.4 MB int8.

Swap to a smaller variant by editing `BACKBONE_NAME`, `HUB_URL`, and `IMAGE_SIZE` in the config cell. See `.claude/skills/tf-efficientnet-lite/SKILL.md`.

## Setup

Dependencies: `tensorflow>=2.13`, `tensorflow-hub>=0.14`, `tensorflow-model-optimization>=0.7`, `matplotlib`.

In [1]:
# Optional install — uncomment on first run.
# %pip install -q "tensorflow>=2.13" "tensorflow-hub>=0.14" "tensorflow-model-optimization>=0.7" matplotlib

In [2]:
import os
# MUST be set BEFORE `import tensorflow`. tensorflow_hub.KerasLayer predates Keras 3
# and cannot consume the new KerasTensor. This routes `tf.keras` back to legacy Keras 2
# (installed as the `tf_keras` package). If this cell was already run with Keras 3 in
# effect, restart the Jupyter kernel and re-run from the top.
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_model_optimization as tfmot

print("TF:", tf.__version__, "| tf-hub:", hub.__version__, "| tfmot:", tfmot.__version__)
try:
    import tf_keras as _tfk; _keras_ver = _tfk.__version__
except ImportError:
    _keras_ver = getattr(tf.keras, "__version__", "unknown")
print("Keras:", _keras_ver, "(expect 2.x under TF_USE_LEGACY_KERAS=1)")
print("GPUs:", tf.config.list_physical_devices("GPU") or "none - CPU training will be slow.")


TF: 2.21.0 | tf-hub: 0.16.1 | tfmot: 0.8.0
Keras: 2.21.0 (expect 2.x under TF_USE_LEGACY_KERAS=1)
GPUs: none - CPU training will be slow.


## Config

Swap the Lite variant by editing `BACKBONE_NAME`, `HUB_URL`, and `IMAGE_SIZE` together (see the skill file). The legacy `tfhub.dev` URL still resolves through the Kaggle Models redirect; if download fails, switch to `https://www.kaggle.com/models/tensorflow/efficientnet/tensorFlow2/lite4-feature-vector/versions/2`.

In [ ]:
DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

BACKBONE_NAME = "efficientnet_lite4"
HUB_URL = "https://tfhub.dev/tensorflow/efficientnet/lite4/feature-vector/2"
IMAGE_SIZE = 300          # Lite4 native resolution

BATCH_SIZE = 16           # 300-px inputs — keep batch small
EPOCHS_HEAD = 1           # head warmup (frozen backbone)
EPOCHS_FULL = 10          # full fine-tune
EPOCHS_QAT  = 3           # QAT fine-tune (short, small LR)

LR_HEAD = 1e-3
LR_FULL = 1e-4
LR_QAT  = 5e-5

REP_DATASET_SAMPLES = 200 # int8 calibration sample count
SEED = 42

tf.keras.utils.set_random_seed(SEED)

## Data

`data_preparation.ipynb` already populated `data/{train,val,test}/<scientific_name>/`. Class names are derived from the directory listing at runtime — do **not** reuse the stale `outputs/class_names.json` from the DINOv2 pipeline (it has fewer classes than the current data directory).

EfficientNet-Lite expects inputs in `[0, 1]`.

In [4]:
def load_split(split, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        DATA_DIR / split,
        image_size=(IMAGE_SIZE, IMAGE_SIZE),
        batch_size=BATCH_SIZE,
        label_mode="int",
        shuffle=shuffle,
        seed=SEED,
    )

raw_train = load_split("train", shuffle=True)
raw_val   = load_split("val",   shuffle=False)
raw_test  = load_split("test",  shuffle=False)

class_names = raw_train.class_names
num_classes = len(class_names)
print(f"{num_classes} classes. First few: {class_names[:5]}")

normalize = tf.keras.layers.Rescaling(1.0 / 255.0)
augment = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.05),
        tf.keras.layers.RandomZoom(0.05),
    ],
    name="augment",
)

def prep(ds, training=False):
    ds = ds.map(lambda x, y: (normalize(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.cache().prefetch(tf.data.AUTOTUNE)

train_ds = prep(raw_train, training=True)
val_ds   = prep(raw_val)
test_ds  = prep(raw_test)

Found 12111 files belonging to 88 classes.
Found 2548 files belonging to 88 classes.
Found 2679 files belonging to 88 classes.
88 classes. First few: ['Actitis_macularius', 'Allograpta_obliqua', 'Apis_mellifera', 'Arenaria_melanocephala', 'Argyranthemum_foeniculaceum']



## Build the model

EfficientNet-Lite4 feature vector (TF Hub) + Dropout + Dense logits head.

In [5]:
def build_model(num_classes, trainable_backbone=False):
    inputs = tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3), name="image")
    backbone = hub.KerasLayer(HUB_URL, trainable=trainable_backbone, name=BACKBONE_NAME)
    features = backbone(inputs)
    x = tf.keras.layers.Dropout(0.2, name="dropout")(features)
    outputs = tf.keras.layers.Dense(num_classes, name="classifier")(x)
    return tf.keras.Model(inputs, outputs, name=f"{BACKBONE_NAME}_classifier")

model = build_model(num_classes, trainable_backbone=False)
model.summary()

Model: "efficientnet_lite4_classifier"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 image (InputLayer)          [(None, 300, 300, 3)]     0         
                                                                 
 efficientnet_lite4 (KerasL  (None, 1280)              11837936  
 ayer)                                                           
                                                                 
 dropout (Dropout)           (None, 1280)              0         
                                                                 
 classifier (Dense)          (None, 88)                112728    
                                                                 
Total params: 11950664 (45.59 MB)
Trainable params: 112728 (440.34 KB)
Non-trainable params: 11837936 (45.16 MB)
_________________________________________________________________


## Phase 1 — head warmup (frozen backbone)

In [6]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(LR_HEAD),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="acc")],
)
hist_head = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD)

Epoch 1/3



757/757 [==============================] - 354s 465ms/step - loss: 1.2532 - acc: 0.6884 - val_loss: 0.7800 - val_acc: 0.7786
Epoch 2/3
757/757 [==============================] - 333s 440ms/step - loss: 0.6925 - acc: 0.8022 - val_loss: 0.7087 - val_acc: 0.7951
Epoch 3/3
757/757 [==============================] - 339s 448ms/step - loss: 0.5534 - acc: 0.8404 - val_loss: 0.6949 - val_acc: 0.7983


## Phase 3 — Quantization-aware fine-tuning (best-effort)

`tfmot.quantization.keras.quantize_model` inserts fake-quant ops so weights learn to tolerate int8 rounding. Hub-wrapped backbones are usually opaque to tfmot, so the block below **tries** full-model QAT first and falls back to head-only fake-quant if that fails. Either way, the int8 `.tflite` deliverable comes from the full-integer PTQ calibration at conversion time in the next phase — QAT is a tuning step on top of that.

In [1]:
quantize_model = tfmot.quantization.keras.quantize_model
quantize_annotate_layer = tfmot.quantization.keras.quantize_annotate_layer
quantize_apply = tfmot.quantization.keras.quantize_apply

try:
    qat_model = quantize_model(model)
    print("Full-model QAT annotation succeeded.")
except Exception as err:
    print(f"Full-model QAT not supported here ({type(err).__name__}: {err}).")
    print("Falling back to head-only fake-quant; backbone will rely on PTQ calibration.")

    inputs = tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3), name="image")
    backbone = model.get_layer(BACKBONE_NAME)
    features = backbone(inputs)
    dropout = tf.keras.layers.Dropout(0.2, name="dropout_q")(features)
    head_layer = tf.keras.layers.Dense(num_classes, name="classifier_q")
    head_annotated = quantize_annotate_layer(head_layer)(dropout)
    annotated = tf.keras.Model(inputs, head_annotated)
    # Seed the new head from the trained float weights so QAT starts at the float optimum.
    annotated.get_layer("classifier_q").set_weights(
        model.get_layer("classifier").get_weights()
    )
    qat_model = quantize_apply(annotated)

qat_model.compile(
    optimizer=tf.keras.optimizers.Adam(LR_QAT),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="acc")],
)
hist_qat = qat_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_QAT)

qat_sim_loss, qat_sim_acc = qat_model.evaluate(test_ds, verbose=0)
print(f"QAT (float-sim) test accuracy: {qat_sim_acc:.4f}")

NameError: name 'tfmot' is not defined

## Phase 4 — Full-integer INT8 TFLite export

Full-integer conversion uses a **representative dataset** (samples from the training distribution) to calibrate activation ranges. Both input and output tensors are int8 so the model is directly consumable by edge TFLite runtimes without float pre/post-processing.

Calibration draws only from `train_ds` — the test split stays uncontaminated.

In [ ]:
def representative_dataset():
    for images, _ in train_ds.unbatch().batch(1).take(REP_DATASET_SAMPLES):
        yield [tf.cast(images, tf.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(qat_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_bytes = converter.convert()
tflite_path = OUTPUT_DIR / f"{BACKBONE_NAME}_int8.tflite"
tflite_path.write_bytes(tflite_bytes)
print(f"Wrote {tflite_path} ({len(tflite_bytes) / 1024:.1f} KB)")

## Phase 5 — Verify the INT8 TFLite on the test set

Feeds the held-out test split through the exported interpreter, quantizing each float image into int8 using the input tensor's scale/zero-point. Reports TFLite accuracy vs. the Keras float model.

In [ ]:
interpreter = tf.lite.Interpreter(model_content=tflite_bytes)
interpreter.allocate_tensors()
inp_detail = interpreter.get_input_details()[0]
out_detail = interpreter.get_output_details()[0]
in_scale, in_zero = inp_detail["quantization"]
print(f"Input:  dtype={inp_detail['dtype'].__name__} scale={in_scale} zero={in_zero}")
print(f"Output: dtype={out_detail['dtype'].__name__}")

correct = 0
total = 0
for images, labels in test_ds.unbatch():
    x = tf.expand_dims(images, 0)  # normalized float32 in [0, 1]
    x_int8 = np.round(x.numpy() / in_scale + in_zero).clip(-128, 127).astype(np.int8)
    interpreter.set_tensor(inp_detail["index"], x_int8)
    interpreter.invoke()
    logits = interpreter.get_tensor(out_detail["index"])
    pred = int(np.argmax(logits, axis=-1)[0])
    correct += int(pred == int(labels))
    total += 1

tflite_test_acc = correct / total
print(f"INT8 TFLite test accuracy: {tflite_test_acc:.4f}  ({correct}/{total})")
print(f"Float  test accuracy:      {float_test_acc:.4f}")
print(f"Delta (TFLite - float):    {tflite_test_acc - float_test_acc:+.4f}")

## Phase 6 — Save metadata + training curves

In [ ]:
metadata = {
    "model_name": BACKBONE_NAME,
    "hub_url": HUB_URL,
    "image_size": IMAGE_SIZE,
    "num_classes": num_classes,
    "class_names": class_names,
    "float_test_accuracy": float(float_test_acc),
    "qat_sim_test_accuracy": float(qat_sim_acc),
    "tflite_test_accuracy": float(tflite_test_acc),
    "tflite_path": str(tflite_path),
    "tflite_size_kb": round(len(tflite_bytes) / 1024, 1),
    "input_dtype": "int8",
    "output_dtype": "int8",
    "representative_dataset_samples": REP_DATASET_SAMPLES,
}
meta_path = OUTPUT_DIR / f"{BACKBONE_NAME}_metadata.json"
meta_path.write_text(json.dumps(metadata, indent=2))
print(json.dumps(metadata, indent=2))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for label, hist in [("head", hist_head), ("full", hist_full), ("qat", hist_qat)]:
    axes[0].plot(hist.history["loss"], label=f"train {label}")
    axes[0].plot(hist.history["val_loss"], "--", label=f"val {label}")
    axes[1].plot(hist.history["acc"], label=f"train {label}")
    axes[1].plot(hist.history["val_acc"], "--", label=f"val {label}")
axes[0].set_title("Loss");     axes[0].set_xlabel("epoch")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("epoch")
for ax in axes:
    ax.legend(fontsize=7); ax.grid(alpha=0.3)
fig.tight_layout()
curves_path = OUTPUT_DIR / f"{BACKBONE_NAME}_training_curves.png"
fig.savefig(curves_path, dpi=150)
plt.show()
print(f"Saved {curves_path}")

## Deploying to the Arduino UNO Q

Copy `outputs/efficientnet_lite4_int8.tflite` and `outputs/efficientnet_lite4_metadata.json` to the device. The UNO Q's Linux side runs Python 3, so `tflite_runtime` is the minimal inference dependency:

```bash
pip install tflite-runtime
```

Read `metadata["class_names"]` at runtime so on-device index→label mappings stay aligned with training. The `seashore` negative class is already handled upstream by `TourGuide_Agent/demo.py::NEGATIVE_CLASSES`, so no extra logic is needed on the device side.

A follow-up would swap `TourGuide_Agent/classifier.py` from DINOv2 to a `tflite_runtime.interpreter.Interpreter`-backed implementation — out of scope for this notebook.